In [1]:
import re
import time
import random
import numpy as np
import pandas as pd
import speech_recognition as sr
import pyttsx3
import threading
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import joblib

In [2]:
# -----------------------
# 1) Load & preprocess
# -----------------------
file_path = "data/csv/Data_tanggapan_positif.xlsx"

df_krisis = pd.read_excel(file_path, sheet_name="Krisis")
df_tidak = pd.read_excel(file_path, sheet_name="Tidak Krisis")

# Pastikan nama kolom konsisten
df_krisis = df_krisis.rename(columns={"Kalimat": "text", "Respon": "tanggapan"})
df_tidak = df_tidak.rename(columns={"Kalimat": "text", "Respon": "tanggapan"})

df_krisis["label"] = 1
df_tidak["label"] = 0

# Gabungkan
df = pd.concat([df_krisis, df_tidak], ignore_index=True)

# Bersihkan data kosong
df = df.dropna(subset=["text", "tanggapan"])
df["text"] = df["text"].astype(str).str.strip()
df["tanggapan"] = df["tanggapan"].astype(str).str.strip()

def preprocess_text(s: str):
    s = str(s)
    s = s.replace("“", '"').replace("”", '"').replace("—", " ").replace("\n", " ")
    s = s.lower().strip()
    s = re.sub(r'[^0-9a-z\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s)
    return s

df["text_proc"] = df["text"].apply(preprocess_text)

In [3]:
# -----------------------
# 2) Train / validation / test split
# -----------------------
X = df["text_proc"]
y = df["label"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1, stratify=y_train_full, random_state=42
)

In [5]:
# -----------------------
# 3) Vectorizer + Model
# -----------------------
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), max_features=10000, sublinear_tf=True, min_df=2
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

models = {
    "LogReg": LogisticRegression(class_weight="balanced", solver="liblinear", max_iter=2000),
    "LGBM": LGBMClassifier(class_weight="balanced", random_state=42, n_estimators=300),
    "XGBoost": XGBClassifier(
        use_label_encoder=False, 
        eval_metric='logloss', 
        scale_pos_weight= (y_train.value_counts()[0] / y_train.value_counts()[1]),
        random_state=42,
        n_estimators=300
    )
}

model_scores = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    probs = model.predict_proba(X_val_tfidf)[:, 1]
    preds = (probs >= 0.5).astype(int)
    f1 = f1_score(y_val, preds)
    model_scores[name] = {"model": model, "f1": f1}
    print(f"{name} F1 Score: {f1:.4f}")

LogReg F1 Score: 0.9669
[LightGBM] [Info] Number of positive: 673, number of negative: 720
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002185 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2995
[LightGBM] [Info] Number of data points in the train set: 1393, number of used features: 136
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

d:\Projects\Robot Pencegah Bunuh Diri\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Projects\Robot Pencegah Bunuh Diri\.venv\lib\site-packages\xgboost\core.py:158: UserWarning: [09:27:24] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost F1 Score: 0.9796


In [6]:
# -----------------------
# 4b) Pilih model terbaik
# -----------------------
best_model_name = max(model_scores, key=lambda x: model_scores[x]["f1"])
best_model = model_scores[best_model_name]["model"]
best_f1 = model_scores[best_model_name]["f1"]
print(f"🏆 Best model: {best_model_name} (F1={best_f1:.4f})")

🏆 Best model: LGBM (F1=0.9865)


In [7]:
# -----------------------
# 4c) Tuning threshold untuk best model
# -----------------------
probs_val = best_model.predict_proba(X_val_tfidf)[:, 1]

best = {"thresh": 0.5, "recall": 0.0, "precision": 0.0, "f1": 0.0}
for t in np.linspace(0.1, 0.95, 85):
    preds = (probs_val >= t).astype(int)
    r = recall_score(y_val, preds)
    p = precision_score(y_val, preds, zero_division=0)
    f = f1_score(y_val, preds, zero_division=0)
    if r > best["recall"] - 1e-9 and (r > best["recall"] or p > best["precision"] * 0.7):
        best = {"thresh": t, "recall": r, "precision": p, "f1": f}

chosen_threshold = best["thresh"]
print(
    f"✅ Final Threshold: {chosen_threshold:.3f} "
    f"(recall={best['recall']:.3f}, prec={best['precision']:.3f}, f1={best['f1']:.3f})"
)

d:\Projects\Robot Pencegah Bunuh Diri\.venv\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ Final Threshold: 0.353 (recall=0.987, prec=1.000, f1=0.993)


In [9]:
# -----------------------
# 4d) Simpan model & vectorizer
# -----------------------
joblib.dump(best_model, "models/languange/best_model.pkl")
joblib.dump(vectorizer, "models/languange/vectorizer.pkl")
print("💾 Best model & vectorizer saved to disk.")

💾 Best model & vectorizer saved to disk.


In [ ]:
# -----------------------
# 5) Keyword fallback
# -----------------------
CRISIS_KEYWORDS = [
    # ✨ Frasa langsung
    "bunuh diri", "saya mau mati", "saya mati", "bunuh", "ingin mati", "ingin bunuh diri",
    "tidak ingin hidup", "sudah tidak kuat", "sudah tidak sanggup",
    "mati saja", "selesai saja", "akhiri hidup", "putus asa",
    "menyakiti diri", "mengakhiri hidup", "sudah tidak ada harapan",
    "sudah ingin mati", "capek hidup",

    # 💔 Variasi penulisan & ejaan
    "gw mau mati", "gue mau mati", "pengen mati", "pgn mati", "pingin mati",
    "udah ga kuat", "gak kuat lagi", "gk kuat", "ga kuat",
    "cape hidup", "capee hidup", "udah cape", "sudah capek",
    "gak sanggup lagi", "udah nyerah", "nyerah aja",

    # 🥀 Kalimat tidak langsung
    "hidup gak ada artinya", "hidup gak guna", "hidup sia sia",
    "aku pengen hilang", "ingin hilang", "pengen ngilang", "ingin pergi selamanya",
    "mending mati", "lebih baik mati", "biar aku mati aja",
    "ingin tidur selamanya", "ingin berhenti hidup",

    # ⚠️ Perilaku menyakiti diri
    "lukai diri", "melukai diri", "sayat", "nyakitin diri", "self harm",
    "aku menyakiti diri", "aku pengen nyakitin diri", "pengen sayat",
    "pengen nyakitin badan",

    # 😞 Kalimat hopeless
    "aku gak berharga", "aku gagal", "semuanya percuma",
    "hidup ini sia sia", "aku menyerah", "aku nyerah", "udah gak ada harapan",
    "gak ada gunanya hidup"
]

CRISIS_KEYWORDS = [preprocess_text(k) for k in CRISIS_KEYWORDS]

def contains_crisis_keyword(s_proc: str):
    for k in CRISIS_KEYWORDS:
        if k in s_proc:
            return True, k
    return False, None

In [ ]:
# -----------------------
# 6) Classification + tanggapan
# -----------------------
def classify_text(text: str):
    s_proc = preprocess_text(text)

    # keyword fallback
    kw_match, kw = contains_crisis_keyword(s_proc)
    if kw_match:
        responses = df_krisis["tanggapan"].dropna().tolist()
        response = random.choice(responses)
        return {
            "input": text,
            "label": "Krisis",
            "prob_crisis": 1.0,
            "reason": f"keyword:{kw}",
            "respon": response,
        }

    if len(s_proc.split()) <= 2:
        return {
            "input": text,
            "label": "Netral",
            "prob_crisis": None,
            "reason": "short_text",
            "respon": "Saya mendengarkan, silakan ceritakan",
        }

    v = vectorizer.transform([s_proc])
    prob = float(clf.predict_proba(v)[0, 1])
    label = "Krisis" if prob >= chosen_threshold else "Tidak Krisis"

    if label == "Krisis":
        responses = df_krisis["tanggapan"].dropna().tolist()
    else:
        responses = df_tidak["tanggapan"].dropna().tolist()

    response = random.choice(responses) if responses else "Saya mendengarkan kamu, tetap semangat ya 💙"

    return {
        "input": text,
        "label": label,
        "prob_crisis": prob,
        "reason": "model",
        "respon": response,
    }

In [ ]:
# -----------------------
# 7) TTS (non-blocking, coba pilih voice Indonesia)
# -----------------------
def speak_text(text):
    def run():
        engine = pyttsx3.init()
        engine.setProperty("rate", 160)
        engine.setProperty("volume", 1.0)

        # Cari voice bahasa Indonesia
        voices = engine.getProperty("voices")
        voice_id = None
        for v in voices:
            if "indonesia" in v.name.lower() or "id" in v.id.lower():
                voice_id = v.id
                break

        if voice_id:
            engine.setProperty("voice", voice_id)
        else:
            print("⚠️ Voice Indonesia tidak ditemukan, pakai default voice.")

        engine.say(text)
        engine.runAndWait()

    threading.Thread(target=run, daemon=True).start()

# -----------------------
# 8) Speech recognition loop
# -----------------------
recognizer = sr.Recognizer()
mic = sr.Microphone()

recognizer.dynamic_energy_threshold = True
recognizer.energy_threshold = 300
recognizer.pause_threshold = 0.8

print("\n🎤 Sistem siap mendengarkan (ucapkan 'exit' untuk berhenti).\n")
with mic as source:
    recognizer.adjust_for_ambient_noise(source, duration=1.0)
    print("✅ Kalibrasi selesai, mulai mendengarkan...")

def callback(recognizer, audio):
    try:
        text = recognizer.recognize_google(audio, language="id-ID")
        print(f"\n🗣️ Anda berkata: {text}")

        if text.strip().lower() == "exit":
            print("🚪 Keluar dari program...")
            stop_listening(wait_for_stop=False)
            return

        result = classify_text(text)
        print("📊 Prediksi:", result["label"], f"(Prob: {result['prob_crisis']})")
        print("💡 Tanggapan:", result["respon"])

        # 🔊 Ucapkan tanggapan dengan suara
        speak_text(result["respon"])

    except sr.UnknownValueError:
        print("❌ Tidak bisa mengenali suara.")
    except sr.RequestError as e:
        print(f"⚠️ Error STT: {e}")

stop_listening = recognizer.listen_in_background(mic, callback, phrase_time_limit=10)

try:
    while True:
        time.sleep(0.1)
except KeyboardInterrupt:
    print("⛔ Dihentikan oleh user.")
    stop_listening(wait_for_stop=False)
